# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the [FAIR² Mental Health Survey Dataset from Kilifi County, Kenya](https://doi.org/10.71728/senscience.vcs2-05nj) using the `mlcroissant` library. All references to dataset entities—record sets, fields, and columns—are made using their `@id` fields, in line with FAIR and Croissant best practices.

### Dataset Source
The data is described and accessed via a Croissant schema URL:

https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`. This section loads the Croissant schema, prints basic dataset information, and prepares for data exploration.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Show the dataset metadata
print(f"Dataset Title: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")
print(f"Version: {dataset.metadata.version}")
print(f"Published: {dataset.metadata.datePublished}")
print(f"License: {dataset.metadata.license}")

## 2. Data Overview
Review available record sets, fields, and their IDs (`@id`s).
We'll print all record sets (`@id`, name, and description) and all fields for each record set.

In [ ]:
record_sets = dataset.metadata.record_sets
record_set_ids = []

print("Available record sets in the dataset:")
for rs in record_sets:
    print(f"  - @id: {rs.id}\n    name: {rs.name}")
    desc = rs.description if hasattr(rs, 'description') else None
    if desc:
        print(f"    description: {desc}")
    print(f"    Has {len(rs.fields)} fields.")
    record_set_ids.append(rs.id)
    print("    Fields:")
    for field in rs.fields:
        print(f"      * @id: {field.id} - name: {field.name} (type: {field.data_type})")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame.

Below, we load the data from all primary record sets into pandas DataFrames for further exploration. Use the `@id` values shown above to reference each record set and field.

In [ ]:
# Extract records from all record sets into DataFrames
dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

# Display the columns of each DataFrame
for record_set_id in record_set_ids:
    print(f"Record Set @id: {record_set_id}")
    print("Columns:", dataframes[record_set_id].columns.tolist())
    print("First few rows:")
    display(dataframes[record_set_id].head(2))
    print("-"*40)


## 4. Exploratory Data Analysis (EDA)
Apply EDA including filtering records, normalizing numeric fields, and grouping data. All field references use their `@id` names.

Let's choose a main survey record set with numeric mental health scores (such as GAD-7 or PHQ-9 if present), filter on score, normalize, and group by a demographic field.

In [ ]:
# Identify relevant record sets and fields for mental health scores
# Use printed record set and field IDs from above cell

# For this dataset, main survey data is likely in the first record set:
main_record_set_id = record_set_ids[0]
df = dataframes[main_record_set_id]

# Attempt to automatically find GAD-7, PHQ-9, and a demographic field by searching columns
numeric_score_fields = [col for col in df.columns if any(x in col.lower() for x in ["gad", "phq", "psq"]) and df[col].dtype in ["int64", "float64", "object"]]
if not numeric_score_fields:
    # fallback: use any numeric column
    numeric_score_fields = df.select_dtypes(include=['number']).columns.tolist()

if numeric_score_fields:
    numeric_field_id = numeric_score_fields[0]
else:
    raise ValueError("No suitable numeric fields found for EDA.")

print(f"Selected numeric field for analysis: {numeric_field_id}")

# Choose a grouping field (demographics)
group_fields = [col for col in df.columns if any(x in col.lower() for x in ["gender", "age", "village", "education", "marital"])]
group_field_id = group_fields[0] if group_fields else None
if group_field_id:
    print(f"Selected grouping field: {group_field_id}")

# Remove missing values and convert to numeric (if necessary)
df = df.copy()
df = df[pd.to_numeric(df[numeric_field_id], errors='coerce').notna()]
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

# Filtering: e.g., select rows where numeric_field_id > 5
threshold = 5
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}:")
display(filtered_df.head())

# Normalization
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Grouped mean by demographic field if available
if group_field_id:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"Grouped mean of {numeric_field_id} by {group_field_id}:")
    display(grouped_df.head())

## 5. Visualization
Visualize the distribution of the chosen mental health score, and explore group-wise differences if relevant. All fields are referenced by `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

# Distribution of the selected numeric field
plt.figure(figsize=(8, 5))
sns.histplot(df[numeric_field_id], bins=15, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Frequency")
plt.show()

# Boxplot by group field if available
if group_field_id:
    plt.figure(figsize=(8, 5))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.show()

## 6. Conclusion
In this notebook, we demonstrated loading, exploring, and processing the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using the `mlcroissant` library. 

Key steps included:
- Loading the Croissant schema and finding all available record sets and fields by their `@id`.
- Extracting main survey data, identifying and filtering relevant numeric mental health indicators using only `@id`-based field references.
- Visualizing data distributions and group statistics (e.g., by gender or education level).

This approach supports FAIR and reproducible research utilizing the AI-ready Croissant data ecosystem.